In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import yaml

In [ ]:
import os
os.chdir('../../../')

In [ ]:
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

## (1) Knowledge source

In [ ]:
palette = {
    'OLS'      : '#8B1E1E',  # deep muted red
    'Ln + OLS' : '#1F5AA6',  # dark blue
    'Poisson'  : '#2E7D32',  # dark green
    'ratio'    : '#4D4D4D'   # dark gray
}

# ---------- hatch styles ----------
hatch_map = {
    'OLS'      : '----',
    'Ln + OLS' : '....',
    'Poisson'  : '////',
    'ratio'    : '--'
}

# ---------- data ----------
cats = ['Domestic', 'Foreign', 'U.S.', 'Non-U.S.']

coef_ols  = np.array([0.988, 0.277, 0.197, 0.265]); se_ols  = np.array([0.073, 0.023, 0.021, 0.023])
coef_ln   = np.array([0.799, 0.571, 0.307, 0.503]); se_ln   = np.array([0.053, 0.044, 0.031, 0.041])
coef_pois = np.array([0.478, 0.431, 0.285, 0.534]); se_pois = np.array([0.113, 0.130, 0.136, 0.132])

coef_pct = np.array([0.052, 0.011, 0.001, 0.012]); se_pct  = np.array([0.017, 0.009, 0.005, 0.008]) # need update

# ---------- font sizes (ENLARGED) ----------
FS_TICK   = 15   # axis tick labels
FS_LABEL  = 15   # axis titles
FS_ANN    = 11   # numbers on bars
FS_LEGEND = 14   # legend text


# ---------- annotation helpers (auto repel) ----------
def annotate_bars_auto(ax, xpos, coeffs, errs, texts=None, gap_frac=0.02, fontsize=FS_ANN):
    """
    Draw labels at their default positions and return the text artists for a later collision-resolution pass.
    """
    if texts is None:
        texts = []

    ymin, ymax = ax.get_ylim()
    span = ymax - ymin

    for x_i, c, e in zip(xpos, coeffs, errs):
        y = c + e + span * gap_frac
        if y > ymax:  # keep annotation inside the plot
            y = ymax - span * gap_frac

        t = ax.text(
            x_i, y, f'{c:.3f}',
            ha='center', va='bottom', fontsize=fontsize,
            bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=0.15)
        )
        texts.append(t)

    return texts


def repel_texts(ax, texts, max_iter=80, y_step_px=6, pad=1.02):
    """
    Auto-adjust labels: if text boxes overlap, move the later label upward only (pixel units).
    """
    fig = ax.figure
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    def get_bboxes():
        return [t.get_window_extent(renderer).expanded(pad, pad) for t in texts]

    ymin, ymax = ax.get_ylim()

    for _ in range(max_iter):
        moved = False
        bboxes = get_bboxes()

        for i in range(len(texts)):
            for j in range(i):
                if bboxes[i].overlaps(bboxes[j]):
                    x_disp, y_disp = ax.transData.transform(texts[i].get_position())

                    # Move upward only; do not apply any horizontal offset.
                    y_disp += y_step_px

                    x_new, y_new = ax.transData.inverted().transform((x_disp, y_disp))
                    y_new = min(y_new, ymax - (ymax - ymin) * 0.01)   # Keep a small margin from the top boundary.

                    texts[i].set_position((x_new, y_new))
                    moved = True
                    break

        if not moved:
            break

        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()


def nudge_text_by_value(ax, texts, value_str, dx_px=0, dy_px=0, occurrence=0, min_y=None):
    """
    Nudge a label by its displayed text (e.g., '0.571'); use occurrence to select among duplicates.
    dx_px/dy_px are pixel offsets (dy_px > 0 moves up, dy_px < 0 moves down).
    """
    matches = [t for t in texts if t.get_text() == value_str]
    if len(matches) <= occurrence:
        return

    t = matches[occurrence]
    x, y = t.get_position()
    x_disp, y_disp = ax.transData.transform((x, y))
    x_new, y_new = ax.transData.inverted().transform((x_disp + dx_px, y_disp + dy_px))
    if min_y is not None:
        y_new = max(y_new, min_y)
    t.set_position((x_new, y_new))


# ---------- Panel A: levels ----------
x, w = np.arange(len(cats)), 0.25
fig1, ax1 = plt.subplots(figsize=(7, 6))

bars_ols = ax1.bar(
    x - w, coef_ols, w, yerr=1.645 * se_ols,
    color=palette['OLS'], capsize=4, label='OLS',
    edgecolor='white', linewidth=1.2, hatch=hatch_map['OLS']
)
bars_ln = ax1.bar(
    x, coef_ln, w, yerr=1.645 * se_ln,
    color=palette['Ln + OLS'], capsize=4, label='Ln + OLS',
    edgecolor='white', linewidth=1.2, hatch=hatch_map['Ln + OLS']
)
bars_poi = ax1.bar(
    x + w, coef_pois, w, yerr=1.645 * se_pois,
    color=palette['Poisson'], capsize=4, label='Poisson',
    edgecolor='white', linewidth=1.2, hatch=hatch_map['Poisson']
)

# dynamic y-limit (optional; you override with set_ylim below)
top1 = (np.concatenate([
    coef_ols  + 1.645 * se_ols,
    coef_ln   + 1.645 * se_ln,
    coef_pois + 1.645 * se_pois
]).max() + 0.05)
ax1.set_ylim(top=top1)

# Keep your original limits (set before annotations so auto-adjust uses a stable transform).
ax1.set_ylim(bottom=-0.25, top=1.75)

# write numbers + auto repel
texts = []
texts = annotate_bars_auto(ax1, x - w, coef_ols,  1.645 * se_ols, texts=texts)
texts = annotate_bars_auto(ax1, x,     coef_ln,   1.645 * se_ln,  texts=texts)
texts = annotate_bars_auto(ax1, x + w, coef_pois, 1.645 * se_pois, texts=texts)
repel_texts(ax1, texts)

# fine-tune the two foreign labels
nudge_text_by_value(ax1, texts, '0.571', dy_px=-8)
nudge_text_by_value(ax1, texts, '0.431', dy_px=-8)
# shift OLS labels slightly left to avoid touching the middle bars
nudge_text_by_value(ax1, texts, '0.799', dx_px=3)
nudge_text_by_value(ax1, texts, '0.478', dx_px=3)
nudge_text_by_value(ax1, texts, '0.277', dx_px=-3)
nudge_text_by_value(ax1, texts, '0.197', dx_px=-3)
nudge_text_by_value(ax1, texts, '0.265', dx_px=-3)

# styling
ax1.axhline(0, ls='--', c='grey')
ax1.set_xticks(x)
ax1.set_xticklabels(cats, fontsize=FS_TICK)
ax1.set_ylabel('Effect of Entity List', fontsize=FS_LABEL, fontweight='bold')

ax1.tick_params(axis='x', labelsize=FS_TICK)
ax1.tick_params(axis='y', labelsize=FS_TICK)

ax1.legend(frameon=False, loc='upper left', fontsize=FS_LEGEND)

fig1.tight_layout()
fig1.savefig(dataset_config['path_figure'] + 'CN_CN/main_domestic_number.png', dpi=600)


# ---------- Panel B: ratios ----------
fig2, ax2 = plt.subplots(figsize=(7, 6))

bars_ratio = ax2.bar(
    x, coef_pct, width=0.5, yerr=1.645 * se_pct,
    color=palette['ratio'], capsize=4, label='ratio',
    edgecolor='white', linewidth=1.2, hatch=hatch_map['ratio']
)

# keep your original limits
ax2.set_ylim(bottom=-0.1, top=0.2)

# Panel B keeps the simple annotation helper (unchanged).
def annotate_bars(ax, xpos, coeffs, errs, gap_frac=0.02, fontsize=FS_ANN):
    ymin, ymax = ax.get_ylim()
    span = ymax - ymin
    for x_i, c, e in zip(xpos, coeffs, errs):
        y = c + e + span * gap_frac
        if y > ymax:
            y = ymax - span * gap_frac
        ax.text(x_i, y, f'{c:.3f}', ha='center', va='bottom', fontsize=fontsize)

annotate_bars(ax2, x, coef_pct, 1.645 * se_pct)

ax2.axhline(0, ls='--', c='grey')
ax2.set_xticks(x)
ax2.set_xticklabels(cats, fontsize=FS_TICK)
ax2.set_ylabel('Effect of Entity List', fontsize=FS_LABEL, fontweight='bold')

ax2.tick_params(axis='x', labelsize=FS_TICK)
ax2.tick_params(axis='y', labelsize=FS_TICK)

# Optional legend for Panel B (you previously constructed handles but did not draw a legend)
# handles = [Patch(facecolor=palette[k], edgecolor='white', hatch=hatch_map[k], label=k) for k in palette.keys()]
# ax2.legend(handles=handles, frameon=False, fontsize=FS_LEGEND, loc='upper left')

fig2.tight_layout()
# fig2.savefig(dataset_config['path_figure'] + 'CN_CN/main_domestic_ratio.png', dpi=600)

plt.show()


## (2) Vintage

In [ ]:
# ------------ Data ------------

labels = [
    r"$\bf{Publication\ Year\ (Min)}$" + "\noldest cited\nscientific article",
    r"$\bf{Publication\ Year\ (Max)}$" + "\nlatest cited\nscientific article",
    r"$\bf{Publication\ Year\ (Mean)}$" + "\nmean year of\ncited scientific articles",
    r"$\bf{Publication\ Year\ (Median)}$" + "\nmedian year of\ncited scientific articles"
]

coef = np.array([-1.933, 0.800, 0.927, 0.996])
se   = np.array([0.997, 0.361, 0.356, 0.367])

err  = 1.645 * se   # 90% confidence intervals

# Colors and hatches for the remaining five categories
colors  = ['#8B1E1E', "#1F5AA6",    # Publication Year (Min, Max)
           "#2E7D32", '#460460']               

hatches = ['||||', '////',            # Publication Year (Min, Max)
           '....', '----']


# ------------ Helper function: annotate numerical values ------------
def annotate_bars(ax, ypos, coeffs, errs, gap_frac=0.02):
    """
    Annotate horizontal bars by placing coefficient values to the right
    of each error bar.
    """
    xmin, xmax = ax.get_xlim()
    span = xmax - xmin
    for y_i, c, e in zip(ypos, coeffs, errs):
        x_pos = c + e + span * gap_frac
        if x_pos > xmax:
            x_pos = xmax - span * gap_frac
        ax.text(x_pos, y_i, f'{c:.3f}', va='center', ha='left', fontsize=10)

In [ ]:
# ------------ Plotting ------------
fig, ax = plt.subplots(figsize=(10, 6))
y = np.arange(len(labels))

bars = ax.barh(
    y, coef, xerr=err, height=0.7,
    color=colors, edgecolor='white', linewidth=1.2,
    error_kw={'capsize': 4}
)

# Add hatch patterns to each bar
for bar, hatch in zip(bars, hatches):
    bar.set_hatch(hatch)

# Reference vertical line at zero
ax.axvline(0, ls='--', lw=0.8, c='grey')

# -------- Axis labels and ticks (ENLARGED) --------
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=14)   # y-axis labels
ax.set_xlabel('Effect of Entity List', fontsize=14, fontweight='bold')

ax.tick_params(axis='x', labelsize=14)    # x-axis ticks
ax.tick_params(axis='y', labelsize=14)    # y-axis ticks (redundant but safe)

ax.invert_yaxis()

# Set x-axis limits
ax.set_xlim(left=-15, right=15)

# -------- Numerical annotation on bars (ENLARGED) --------
def annotate_bars(ax, ypos, coeffs, errs, gap_frac=0.02, fontsize=12):
    xmin, xmax = ax.get_xlim()
    span = xmax - xmin
    for y_i, c, e in zip(ypos, coeffs, errs):
        x_pos = c + e + span * gap_frac
        if x_pos > xmax:
            x_pos = xmax - span * gap_frac
        ax.text(
            x_pos, y_i, f'{c:.3f}',
            va='center', ha='left', fontsize=fontsize
        )

annotate_bars(ax, y, coef, err, fontsize=14)

'''
# -------- Legend (ENLARGED) --------
handles = [
    Patch(facecolor='#8B1E1E', edgecolor='white', hatch='----',
          label='Publication Year'),
    Patch(facecolor='#2E7D32', edgecolor='white', hatch='....',
          label='Temporal Range')
]
ax.legend(handles=handles, frameon=False, loc='upper left', fontsize=12)
'''

plt.tight_layout()
fig.savefig(dataset_config['path_figure'] + 'CN_CN/main_recency.png', dpi=600)

plt.show()